# Co-authorship: Collaboration Metrics and Network Structure

This notebook examines how biomedical research is authored: how team sizes have changed, how collaboration has grown, and what the structure of co-authorship looks like among prolific authors.

It is built in two parts for a sound reason. A single co-authorship graph over all 3 million articles is not tractable: author lists range up to roughly 2,929 names, and one large-consortium paper alone generates millions of co-author pairs (k authors make k-choose-2 edges), which would dominate the graph and exhaust memory. So Part A computes collaboration metrics across the full corpus (robust, no graph), and Part B builds an actual co-authorship network on a deliberately scoped subset (post-2014, normal-sized teams, prolific authors only).

Two limits set by the data, carried from the EDA. Author names are not disambiguated: "J Smith" at one institution and "J Smith" at another are the same node, so this is a name-level network, not a person-level one. And affiliation coverage is only reliable after 2014, so the network is restricted to that range.

Runs on the published metadata (author_names is included; no abstracts needed).

## Setup

In [ ]:
import os, glob, collections, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# =====================================================================
# DATA LOADING: pick ONE option (same pattern as 03_eda / 04)
# =====================================================================

# ---- OPTION A: LOCAL (active) ----
ROOT = os.path.dirname(os.getcwd())
DATA_DIR = os.path.join(ROOT, "data", "2_clean")

# ---- OPTION B: KAGGLE (uncomment on Kaggle) ----
# _hits = glob.glob("/kaggle/input/*/**/*.parquet", recursive=True) or glob.glob("/kaggle/input/*/*.parquet")
# assert _hits, "no parquet under /kaggle/input; attach the dataset as input"
# DATA_DIR = os.path.dirname(_hits[0])
# =====================================================================

df = pd.read_parquet(DATA_DIR, columns=["uid", "year", "author_names", "n_authors"])
df = df[df["year"] <= 2025].copy()
print(f"loaded {len(df):,} records")
assert len(df) == df["uid"].nunique(), "duplicate PMIDs; dedup did not run"
df[["year", "n_authors"]].head(3)

# Part A: Collaboration metrics (full corpus)

These use every article and need no graph, so they are robust and fast. They describe how team sizes and collaboration patterns have changed across the full 1994-2025 range.

## 1. Team size over time

Mean and median authors per paper per year, plus the share of solo and large-team papers. The mean is sensitive to the large-consortia tail, so the median is the more stable measure of the typical paper.

In [ ]:
by_year = df.groupby("year")["n_authors"]
metrics = pd.DataFrame({
    "mean":   by_year.mean(),
    "median": by_year.median(),
})
metrics["pct_solo"]  = df.assign(s=df["n_authors"] == 1).groupby("year")["s"].mean() * 100
metrics["pct_large"] = df.assign(l=df["n_authors"] >= 10).groupby("year")["l"].mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
sns.lineplot(data=metrics[["mean", "median"]], dashes=False, markers=True, ax=axes[0])
axes[0].set_title("Authors per paper (mean vs median)"); axes[0].set_xlabel("year"); axes[0].set_ylabel("authors")
sns.lineplot(data=metrics[["pct_solo", "pct_large"]], dashes=False, markers=True, ax=axes[1])
axes[1].set_title("Solo (1 author) vs large (10+ author) papers"); axes[1].set_xlabel("year"); axes[1].set_ylabel("% of papers")
plt.tight_layout(); plt.show()
print(metrics.loc[metrics.index.isin([1995, 2005, 2015, 2025])].round(1).to_string())

**What this shows:** the typical paper (median) gains authors only slowly, but the mean rises faster and more erratically because the large-consortia tail grows. Solo authorship declines steadily while the share of 10+ author papers climbs, confirming the shift toward team science seen in the EDA. The gap between mean and median is itself a signal: it widens as large collaborations become more common.

## 2. Team-size composition

The full distribution of team sizes by band, as a share of each year's papers. This shows not just that teams grew, but which size classes gained and which shrank.

In [ ]:
band = pd.cut(df["n_authors"], [0, 1, 5, 10, 20, 10**9],
              labels=["solo", "2-5", "6-10", "11-20", "21+"])
comp = df.assign(band=band).groupby(["year", "band"], observed=True).size().unstack(fill_value=0)
comp_pct = comp.div(comp.sum(axis=1), axis=0) * 100

comp_pct.plot.area(figsize=(13, 5), alpha=0.8,
                   color=["#d9534f", "#1d6fb8", "#2a9d5c", "#8a5fb0", "#e0853f"])
plt.title("Team-size composition by year (%)"); plt.xlabel("year"); plt.ylabel("% of papers")
plt.legend(title="authors", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout(); plt.show()
print(comp_pct.loc[comp_pct.index.isin([1995, 2005, 2015, 2025])].round(1).to_string())

**What this shows:** the bands redistribute over time. Solo and small (2-5) papers lose share while mid and large bands grow, so the "average" rise is a genuine shift in how research is organised, not just a few mega-papers pulling the mean. The composition view is more honest than the mean alone because it shows where the change actually happens.

# Part B: Co-authorship network (scoped subset)

A real co-authorship graph, built only after excluding the cases that would make it intractable or misleading: large-consortium papers (which create giant cliques and explode the edge count), pre-2014 records (unreliable affiliations), and one-off authors (who add noise without structure). The result is a network of how prolific authors collaborate, not a complete map of all authorship.

## 3. Why the cap is necessary

A paper with k authors generates k-choose-2 co-author pairs. For normal teams this is small, but the consortia tail is catastrophic: a single 2,929-author paper alone would generate over four million edges. The distribution below shows how many papers exceed each author count, justifying the cap applied next.

In [ ]:
import math
print("co-author pairs generated by one paper, by author count:")
for n in [5, 10, 20, 50, 100, 500, 2929]:
    print(f"  {n:>5} authors -> {math.comb(n, 2):>12,} pairs")

print("\nhow many papers exceed each author count:")
for n in [10, 20, 50, 100, 200, 500]:
    cnt = (df["n_authors"] > n).sum()
    print(f"  > {n:>3} authors: {cnt:>9,} papers ({cnt/len(df)*100:.2f}%)")

**What this shows:** the vast majority of papers have modest author counts, but the thin tail above (say) 50 authors would dominate any co-authorship graph through sheer pair count. Excluding them removes a tiny fraction of papers while preventing a few consortia from generating most of the edges. The cap is a deliberate, evidence-based decision, not an arbitrary trim.

## 4. Build the scoped network

The graph keeps post-2014 papers with at most MAX_AUTHORS authors, then restricts to authors with at least MIN_PAPERS papers (the recurring, prolific names) and edges of at least MIN_COLLAB joint papers (removing one-off collaborations). The thresholds are explicit and can be tuned.

In [ ]:
MAX_AUTHORS = 50      # exclude consortia from the pairwise graph (see section 3)
MIN_PAPERS  = 5       # keep recurring authors only
MIN_COLLAB  = 3       # keep author pairs who co-published at least this many times

sub = df[(df["year"] >= 2014) & (df["n_authors"] <= MAX_AUTHORS)]
print(f"papers in scope (post-2014, <= {MAX_AUTHORS} authors): {len(sub):,} "
      f"({len(df) - len(sub):,} excluded)")

pair = collections.Counter()
freq = collections.Counter()
for names in sub["author_names"]:
    if isinstance(names, (list, np.ndarray)):
        u = sorted(set(names))
        freq.update(u)
        pair.update(itertools.combinations(u, 2))
print(f"distinct author names: {len(freq):,}")
print(f"distinct co-author pairs: {len(pair):,}")

In [ ]:
try:
    import networkx as nx
except ImportError:
    print("networkx not installed; run `pip install networkx` to build the graph.")
    nx = None

if nx is not None:
    core = set(a for a, c in freq.items() if c >= MIN_PAPERS)
    G = nx.Graph()
    for (a, b), c in pair.items():
        if a in core and b in core and c >= MIN_COLLAB:
            G.add_edge(a, b, weight=c)
    if G.number_of_nodes():
        comps = sorted(nx.connected_components(G), key=len, reverse=True)
        G = G.subgraph(comps[0]).copy()      # largest connected component for a clean layout

    print(f"scoped network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
    if G.number_of_nodes():
        cent = nx.degree_centrality(G)
        print("\nmost connected authors (most distinct collaborators):")
        for a, v in sorted(cent.items(), key=lambda x: -x[1])[:10]:
            print(f"  {a:<30} {v:.3f}")

**What this shows:** the scoped network captures how prolific, recurring authors collaborate. The most connected nodes are authors who co-publish with many different collaborators. Because names are not disambiguated, a highly central node could in principle be several different people sharing a common name; this is a name-level result and should be read as such.

In [ ]:
if nx is not None and G.number_of_nodes():
    pos = nx.spring_layout(G, k=0.5, seed=42, weight="weight")
    sizes = [200 + 4000 * cent[n] for n in G.nodes()]
    weights = [G[u][v]["weight"] for u, v in G.edges()]
    wmax = max(weights) if weights else 1
    widths = [0.3 + 3.0 * (w / wmax) for w in weights]

    plt.figure(figsize=(14, 11))
    nx.draw_networkx_edges(G, pos, width=widths, alpha=0.2, edge_color="#888")
    nx.draw_networkx_nodes(G, pos, node_size=sizes, node_color="#1d6fb8", alpha=0.85)
    # label only the most central nodes to avoid clutter
    big = dict(sorted(cent.items(), key=lambda x: -x[1])[:20])
    nx.draw_networkx_labels(G.subgraph(big), pos, font_size=7)
    plt.title("Co-authorship network among prolific authors (post-2014, capped)")
    plt.axis("off"); plt.tight_layout(); plt.show()

## 5. Collaboration communities

Community detection finds clusters of authors who collaborate more within the group than outside it: research groups, labs, or tight collaborator circles. The number and size of communities describe how fragmented or interconnected the prolific-author landscape is.

In [ ]:
if nx is not None and G.number_of_nodes():
    from networkx.algorithms.community import greedy_modularity_communities
    communities = list(greedy_modularity_communities(G, weight="weight"))
    sizes = sorted((len(c) for c in communities), reverse=True)
    print(f"found {len(communities)} collaboration communities")
    print(f"largest community sizes: {sizes[:10]}")
    print(f"median community size: {int(np.median(sizes))}")

**What this shows:** the prolific-author network resolves into collaboration communities, each a cluster of authors who work together more than with outsiders. Many small-to-medium communities indicate a landscape of distinct research groups rather than one interconnected mass. Like the network itself, these are name-level groupings.

## 6. Summary and caveats

### What this notebook produced
Part A measured collaboration across the full corpus: team sizes over time (mean, median, and composition by band), and the steady shift from solo to team authorship. Part B built a scoped co-authorship network among prolific, recurring authors (post-2014, normal-sized teams), reported the most connected authors, and decomposed the network into collaboration communities.

### Caveats

**Author names are not disambiguated.** This is the most important limit. "J Smith" is a single node regardless of how many distinct people share that name, so common names are over-merged (inflating their centrality) and a single author who publishes under name variants is split. The network is therefore name-level, not person-level, and centrality rankings should be read with that in mind. Proper disambiguation would require author identifiers (such as ORCID) that this metadata does not contain.

**The network excludes large-consortia papers by design.** Papers above MAX_AUTHORS authors are dropped from the pairwise graph because their k-choose-2 pair count would dominate the edges and exhaust memory. This means the network describes normal-scale collaboration, not mega-consortium science. The excluded fraction is small in paper count but would otherwise be enormous in edge count.

**The network is restricted to post-2014 and to prolific authors.** Affiliation coverage is unreliable before 2014 (EDA), and one-off authors add noise without structure, so both are excluded. The network is thus a view of recurring, recent collaboration, not all authorship. Part A's metrics, which use the full corpus, are the unrestricted view.

**Thresholds are judgement calls.** MAX_AUTHORS, MIN_PAPERS, and MIN_COLLAB shape the network. They are set to produce a readable, meaningful graph and are explicit at the top of section 4, but different values would yield a denser or sparser network. The metrics in Part A do not depend on any of them.

---

## Notebook complete

This notebook measured collaboration trends across the corpus and built a scoped co-authorship network among prolific authors.

**Next up:** proceed to `06_keyword_trends.ipynb` for author-keyword analysis and topic emergence. Start the timeline around 2015, since keywords are near-empty before then (EDA section 7a), and normalize by articles per year so a rising keyword reflects real adoption rather than corpus growth.